# 02 · Analisis kesalahan model

Membandingkan model pembanding (TF-IDF) dengan IndoBERT, lalu melihat kalimat mana yang salah ditebak. Grafiknya bisa dipakai untuk slide presentasi.

Syarat: sudah menjalankan `prepare_data.py`, `train_baseline.py`, dan (jika ada) training IndoBERT.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'training'))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 120)


In [ ]:
from pathlib import Path
from common import TUGAS, muat_model
from evaluasi import terapkan_ambang
from src.intent.decide_level import pilih_maksud
from config.settings import get_thresholds
cfg_t = get_thresholds()
model_tersedia = ['pembanding'] + (['indobert'] if (Path('models/indobert-intent') / 'config.json').exists() else [])
print('Model tersedia:', model_tersedia)


## 1. Perbandingan akurasi

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
hasil = []
for tugas in ('intent', 'emotion'):
    for nama_set, path in (('test', TUGAS[tugas]['processed'] / 'test.csv'), ('uji realistis', TUGAS[tugas]['uji_realistis'])):
        df = pd.read_csv(path)
        for jenis in model_tersedia:
            m = muat_model(tugas, jenis)
            probs = [m.predict_proba(t) for t in df['teks']]
            if tugas == 'intent':
                pred = [pilih_maksud(p, cfg_t['intent']['ambang_darurat']) for p in probs]
            else:
                pred = terapkan_ambang(probs, cfg_t['emosi']['ambang'][jenis])
            hasil.append({'tugas': tugas, 'set': nama_set, 'model': jenis,
                          'akurasi': accuracy_score(df['label'], pred), 'f1_makro': f1_score(df['label'], pred, average='macro')})
tabel = pd.DataFrame(hasil)
tabel.round(3)


In [ ]:
ax = tabel[tabel['set'] == 'test'].pivot(index='tugas', columns='model', values='akurasi').plot.bar(rot=0, ylim=(0, 1), figsize=(7, 3.5), title='Akurasi di data test')
ax.set_ylabel('akurasi'); plt.tight_layout(); plt.show()


## 2. Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
jenis = model_tersedia[-1]
for tugas in ('intent', 'emotion'):
    df = pd.read_csv(TUGAS[tugas]['processed'] / 'test.csv')
    m = muat_model(tugas, jenis)
    pred = [max(p, key=p.get) for p in (m.predict_proba(t) for t in df['teks'])]
    ConfusionMatrixDisplay.from_predictions(df['label'], pred, labels=TUGAS[tugas]['labels'], xticks_rotation=45, cmap='Blues')
    plt.title(f'{tugas} / {jenis}'); plt.tight_layout(); plt.show()


## 3. Kalimat yang salah ditebak

Pola kesalahan menunjukkan data apa yang perlu ditambah. Contoh: kalau keluhan ringan sering menjadi darurat, tambah contoh bantuan yang memakai kata 'sakit'.

In [ ]:
tugas = 'intent'
df = pd.read_csv(TUGAS[tugas]['uji_realistis'])
m = muat_model(tugas, jenis)
df['prediksi'] = [pilih_maksud(m.predict_proba(t), cfg_t['intent']['ambang_darurat']) for t in df['teks']]
df['yakin'] = [round(max(m.predict_proba(t).values()), 2) for t in df['teks']]
df[df['label'] != df['prediksi']]


## 4. Pengaruh ambang darurat

Makin kecil ambang, makin sedikit darurat terlewat, tapi alarm palsu bertambah. Pilih titik yang recall merah-nya minimal 0.95.

In [ ]:
from evaluasi import level_gabungan
from sklearn.metrics import precision_score
df = pd.concat([pd.read_csv(TUGAS['intent']['processed'] / 'test.csv'), pd.read_csv(TUGAS['intent']['uji_realistis'])])
m = muat_model('intent', jenis)
probs = [m.predict_proba(t) for t in df['teks']]
asli = level_gabungan(df['teks'], df['label'])
baris = []
for am in [1.0, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1]:
    lv = level_gabungan(df['teks'], [pilih_maksud(p, am) for p in probs])
    baris.append({'ambang': am, 'recall_merah': recall_score(asli, lv, labels=['merah'], average='macro'),
                  'presisi_merah': precision_score(asli, lv, labels=['merah'], average='macro', zero_division=0)})
pd.DataFrame(baris).round(3)
